# Comparing U-Net vs Res-UNet

In this short notebook we compare the two models trained in the previous notebooks:
- `bestmodel.keras` — the plain **U-Net** (from the U-Net notebook).
- `bestmodel_ResUNet.keras` — the **Res-UNet** with residual blocks (from `03-ResUNet.ipynb`).

Both models were trained on the same **Brain Tumor Segmentation Dataset** with the same 4 classes, so we can evaluate them on the same dataset and compare their metrics (loss, accuracy, mean IoU) side by side. This lets us check, in practice, whether adding residual connections actually improved the model, as we expected from the theory.


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')

if gpus:
    try:
        # Enable memory growth for each detected GPU
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for GPU")
    except RuntimeError as e:
        # Memory growth must be set before initializing the TensorFlow runtime
        print(f"Error setting memory growth: {e}")

## Rebuilding the Evaluation Dataset

We rebuild the same image/mask loading pipeline used during training (same preprocessing, same 256x256 resize, same label encoding), so that both models are evaluated under identical conditions.


In [ ]:
DATASET_PATH = './Brain Tumor Segmentation Dataset/'

In [ ]:
image_base_path = os.path.join(DATASET_PATH, 'image')
mask_base_path = os.path.join(DATASET_PATH, 'mask')
class_dirs = [d for d in os.listdir(image_base_path) if os.path.isdir(os.path.join(image_base_path, d))]

In [ ]:
all_image_paths = []
all_mask_paths = []
all_labels = []

for i in range(len(class_dirs)):
    class_dir = class_dirs[i]  

    image_folder = os.path.join(image_base_path, class_dir)
    mask_folder = os.path.join(mask_base_path, class_dir)

    label = int(class_dir)

    for file_name in os.listdir(image_folder):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            img_path = os.path.join(image_folder, file_name)
            
            base_name, extension = os.path.splitext(file_name)
            mask_filename = f"{base_name}_m{extension}"
            mask_path = os.path.join(mask_folder, mask_filename)
            
            if os.path.exists(mask_path):
                all_image_paths.append(img_path)
                all_mask_paths.append(mask_path)
                all_labels.append(label)

all_image_paths = np.array(all_image_paths)
all_mask_paths = np.array(all_mask_paths)
all_labels = np.array(all_labels)

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (256,256)

In [ ]:
def load_and_preprocess_multiclass(image_path, mask_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    
    mask = tf.where(mask > 128, tf.cast(label, tf.uint8), tf.cast(0, tf.uint8))
    
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')
    
    return img, mask

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((all_image_paths, all_mask_paths, all_labels))
train_dataset = train_dataset.map(load_and_preprocess_multiclass, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

## Loading Both Trained Models

We load the two saved checkpoints — the plain U-Net and the Res-UNet — so we can call `.evaluate()` on each with the exact same dataset.


In [ ]:
model_unet = load_model('bestmodel.keras')
model_resunet = load_model('bestmodel_ResUNet.keras')

## Evaluating Both Models

We run `model.evaluate()` for each model on the same dataset and print the resulting metrics dictionary (loss, accuracy, and any additional metrics such as mean IoU, if the model was compiled with them). Comparing these numbers directly tells us whether the residual blocks in Res-UNet gave a measurable improvement over the plain U-Net on this dataset.


In [ ]:
# metric

In [ ]:
eval_unet = model_unet.evaluate(train_dataset,return_dict=True,verbose=0)
print(eval_unet)

In [ ]:
eval_resunet = model_resunet.evaluate(train_dataset,return_dict=True,verbose=0)
print(eval_resunet)